### EMBEDDING AND VECTOR DB

In [1]:
# Import libraries for embedding generation, vector storage, and array operations.
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

In [2]:
# Define file paths, collection name, and embedding model used throughout the pipeline.
CHUNKS_PATH          = "../data/processed/chunks.json"
CHROMA_PATH          = "../data/processed/chroma_db"
MANIFEST_PATH        = "../data/processed/manifest.json"
PENDING_CHANGES_PATH = "../data/processed/pending_changes.json"
COLLECTION_NAME      = "elte_ik"
EMBEDDING_MODEL      = "all-MiniLM-L6-v2"

In [3]:
# Wrap SentenceTransformer in a class to load the model once and encode text batches.
class EmbeddingPipeline:
    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model = SentenceTransformer(model_name)
        print(f"Loaded model: {model_name}")

    def encode(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=True)

In [4]:
# Load pending changes from notebook 01 (or fall back to full re-embed if missing).
from pathlib import Path

with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks from {CHUNKS_PATH}")

pending_path = Path(PENDING_CHANGES_PATH)
if pending_path.exists():
    with open(pending_path, encoding="utf-8") as f:
        pending = json.load(f)
    print(f"Pending changes: {len(pending['new'])} new, "
          f"{len(pending['updated'])} updated, "
          f"{len(pending['deleted'])} deleted")
else:
    # No pending file -> first run / migration: re-embed everything
    pending = {
        "new":     sorted({c["metadata"].get("source_relative") for c in chunks if c["metadata"].get("source_relative")}),
        "updated": [],
        "deleted": [],
    }
    print(f"No pending_changes.json found -> full re-embed of {len(pending['new'])} files")

# Filter chunks to only those that need (re)embedding
to_embed_paths = set(pending["new"]) | set(pending["updated"])
chunks_to_embed = [c for c in chunks if c["metadata"].get("source_relative") in to_embed_paths]
print(f"Will embed {len(chunks_to_embed)} chunks from {len(to_embed_paths)} files")

if chunks_to_embed:
    pipeline = EmbeddingPipeline()
    texts = [c["content"] for c in chunks_to_embed]
    embeddings = pipeline.encode(texts)
    print(f"Embeddings shape: {embeddings.shape}")
else:
    pipeline = None
    texts = []
    embeddings = None
    print("Nothing to embed.")

Loaded 6112 chunks from ../data/processed/chunks.json
Pending changes: 0 new, 0 updated, 0 deleted
Will embed 0 chunks from 0 files
Nothing to embed.


In [5]:
# One-time migration: purge old integer-ID chunks left over from the pre-incremental schema.
# Safe to re-run — does nothing if old chunks are already gone.

_client = chromadb.PersistentClient(path=CHROMA_PATH)
_col = _client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

_all = _col.get(include=[])
_old_ids = [id_ for id_ in _all["ids"] if "::" not in id_]

if _old_ids:
    _BATCH = 5000
    for _i in range(0, len(_old_ids), _BATCH):
        _col.delete(ids=_old_ids[_i:_i + _BATCH])
    print(f"Migration: deleted {len(_old_ids)} old integer-ID chunks")
else:
    print("Migration: no old integer-ID chunks found, nothing to do")

print(f"Collection now has {_col.count()} documents")

Migration: deleted 6112 old integer-ID chunks
Collection now has 6112 documents


In [6]:
# Incrementally update ChromaDB: delete old IDs for updated/deleted files, upsert new chunks.
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# 1. Load manifest to reconstruct old IDs for deletion
with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

# 2. Build the list of IDs to delete
# For deleted files: use the manifest entry that was just removed (we need it from BEFORE the delete).
# Notebook 01 already removed deleted entries from manifest, so we instead query ChromaDB by source_relative.
ids_to_delete = []

# For updated files: manifest now has the NEW chunk_count, but we need to delete OLD chunks first.
# We use ChromaDB's metadata filter on source_relative — this is the safest approach since the
# old chunk count is no longer in the manifest by the time notebook 02 runs.
files_to_clear = list(set(pending["updated"]) | set(pending["deleted"]))
for src_rel in files_to_clear:
    # Query existing docs for this source_relative and collect their IDs
    existing = collection.get(where={"source_relative": src_rel}, include=[])
    if existing["ids"]:
        ids_to_delete.extend(existing["ids"])

if ids_to_delete:
    BATCH_SIZE = 5000
    for i in range(0, len(ids_to_delete), BATCH_SIZE):
        collection.delete(ids=ids_to_delete[i:i+BATCH_SIZE])
    print(f"Deleted {len(ids_to_delete)} old chunks from ChromaDB")
else:
    print("No old chunks to delete")

# 3. Upsert new + updated chunks
if chunks_to_embed:
    ids   = [c["metadata"]["chunk_id"] for c in chunks_to_embed]
    metas = [c["metadata"] for c in chunks_to_embed]
    embs  = embeddings.tolist()

    BATCH_SIZE = 5000
    for i in range(0, len(ids), BATCH_SIZE):
        collection.upsert(
            ids=ids[i:i+BATCH_SIZE],
            embeddings=embs[i:i+BATCH_SIZE],
            documents=texts[i:i+BATCH_SIZE],
            metadatas=metas[i:i+BATCH_SIZE],
        )
    print(f"Upserted {len(ids)} chunks into '{COLLECTION_NAME}'")

print(f"Collection now has {collection.count()} documents")

# 4. Clean up pending_changes so the next run starts fresh
if pending_path.exists():
    pending_path.unlink()
    print(f"Removed {PENDING_CHANGES_PATH}")

No old chunks to delete
Collection now has 6112 documents
Removed ../data/processed/pending_changes.json


In [9]:
# Run a sample query to verify retrieval returns relevant chunks from the collection.
_pipeline = pipeline if pipeline is not None else EmbeddingPipeline()

query = "what is Notification of accommodation?"
query_emb = _pipeline.encode([query]).tolist()

results = collection.query(query_embeddings=query_emb, n_results=3)
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n--- Result {i+1} (chunk {meta['chunk_id']}, {meta['file_name']}) ---")
    print(doc[:300])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: all-MiniLM-L6-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Result 1 (chunk elte.hu/en/for-non-eea-citizens-with-eu-residence-permits.html::chunk_11, for-non-eea-citizens-with-eu-residence-permits.html) ---
[Case A] Dormitory certificate (stamped and signed by the management); or
[Case A] Proof of student hostel (stamped and signed by the management); or
[Case A] Lease contract , which should contain your and your landlord’s personal data (ID card or passport number, mother’s maiden name), the exact ad

--- Result 2 (chunk linked_pdfs/TAJ.pdf::chunk_6, TAJ.pdf) ---
The accommodation reporting form has to be validated by 
the Regional Directorate of the National Directorate-General 
for Aliens Policing
PLEASE NOTE! Our students can only go to the CLIENT SERVICE 
II. Office, at 1135 Budapest XIII., Szegedi út 35-37., Ground Floor 
(Twin Office Center)

--- Result 3 (chunk linked_pdfs/TAJ.pdf::chunk_5, TAJ.pdf) ---
Required documents
1. TAJ-card request sheet
filled out and signed
2. Authorization document
filled out and signed
Both sides of 